Implementation for OpenSource Clipping untuk Single Truth Source.

Link Github: [Here](https://github.com/NaufalRizqullah/opensource-clipping/tree/main)

# 1. Setup Project

In [6]:
# Clone Project

!rm -rf ./* ./.*
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git .

rm: refusing to remove '.' or '..' directory: skipping './.'
rm: refusing to remove '.' or '..' directory: skipping './..'
Cloning into '.'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 29 (delta 3), reused 26 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 84.84 KiB | 10.60 MiB/s, done.
Resolving deltas: 100% (3/3), done.


## Pilih provider AI

Isi pengaturan ini **sebelum menjalankan sel instalasi dependency**:

- `AI_PROVIDER = "gemini"` mempertahankan alur bawaan dan memerlukan `GOOGLE_API_KEY`.
- `AI_PROVIDER = "vllm"` memakai server vLLM tanpa Google API key.
- **Server yang sudah aktif:** biarkan `START_LOCAL_VLLM = False`, isi `VLLM_BASE_URL` dan `VLLM_MODEL` sesuai server, lalu tambahkan `VLLM_API_KEY` ke Secrets hanya jika autentikasi diperlukan.
- **Server di notebook ini:** isi `START_LOCAL_VLLM = True`. Sel berikut menyiapkan environment aplikasi/server Python 3.12 terpisah, menyalakan vLLM, dan menguji transkrip pendek. Preset awal Gemma 4 Q4 mengikuti [notebook khusus](kaggle-vllm-gemma4.ipynb) dan masih eksperimental.

Server lokal memerlukan Linux x86_64 dan GPU yang didukung (T4 atau SM 7.5+; P100 tidak didukung). Kecukupan memori tetap perlu diuji. Whisper memakai CPU/int8 saat GPU dipakai vLLM; mulai dengan model Whisper kecil dan satu klip. Server eksternal mempertahankan pengaturan Whisper Anda.

`VLLM_CONTEXT_LIMIT` adalah batas konteks total; `VLLM_MAX_TOKENS` harus lebih kecil agar tersedia ruang untuk prompt. `VLLM_TIMEOUT` adalah batas waktu analisis/retry dalam detik. Endpoint eksternal harus menyediakan `/v1` dan `/tokenize` serta dapat diakses dari notebook; `127.0.0.1` menunjuk runtime ini. Lihat [panduan provider](../docs/providers.md) dan [setup Kaggle](../docs/KAGGLE_VLLM.md).

Biarkan server aktif selama clipping. Setelah mengganti provider/server, jalankan ulang sel setup; hentikan server lokal sebelum instalasi ulang atau mengganti model/key.


In [ ]:
# AI provider configuration — edit before installing dependencies
import os
from pathlib import Path
import subprocess
import sys

AI_PROVIDER = "gemini"  # "gemini" (default) or "vllm"
START_LOCAL_VLLM = False  # True: install/start the optional server in this notebook
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8001")
VLLM_MODEL = os.environ.get("VLLM_MODEL", "google/gemma-4-E2B-it-qat-w4a16-ct")
VLLM_CONTEXT_LIMIT = os.environ.get("VLLM_CONTEXT_LIMIT", "16384")
VLLM_MAX_TOKENS = os.environ.get("VLLM_MAX_TOKENS", "4096")
VLLM_TIMEOUT = os.environ.get("VLLM_TIMEOUT", "900")
GPU_INDEX = 0
VLLM_PORT = 8001

if AI_PROVIDER not in {"gemini", "vllm"}:
    raise ValueError('Choose AI_PROVIDER = "gemini" or "vllm".')
if START_LOCAL_VLLM and AI_PROVIDER != "vllm":
    raise ValueError('START_LOCAL_VLLM requires AI_PROVIDER = "vllm".')

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
APP_PYTHON = sys.executable
APP_ENV = None
cache_env = {}
LOCAL_VLLM_PREPARED = False
LOCAL_VLLM_READY = False
print(f"AI provider: {AI_PROVIDER}; local vLLM setup: {START_LOCAL_VLLM}")


In [7]:
# Setup Deno
import os

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!curl -fsSL https://deno.land/install.sh | sh


os.environ["PATH"] += ":/root/.deno/bin"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
sh: 105: cannot open /dev/tty: No such device or address


In [ ]:
if not START_LOCAL_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Application dependencies will be installed in the isolated vLLM setup below.")

## API key dan Secrets

Untuk **Gemini**, tambahkan `GOOGLE_API_KEY` di Colab Secrets (aktifkan akses notebook) atau Kaggle **Add-ons → Secrets**.

Untuk **vLLM**, Google API key tidak diperlukan. Isi `VLLM_API_KEY` hanya jika server memerlukan autentikasi. Setup lokal membuat key internal di memori jika belum tersedia; jangan menggantinya saat server aktif.

`PEXELS_API_KEY` untuk B-roll dan `HF_TOKEN` untuk akses Hugging Face/diarisasi bersifat opsional. Environment variable diprioritaskan sebelum Secrets. Secret opsional yang tidak tersedia tidak menghentikan notebook.


In [ ]:
# Load provider credentials from the environment, Colab Secrets, or Kaggle Secrets
secret_readers = []
try:
    from kaggle_secrets import UserSecretsClient
    secret_readers.append(UserSecretsClient().get_secret)
except ImportError:
    pass
except Exception as exc:
    print(f"Kaggle Secrets unavailable ({type(exc).__name__}); check notebook access.")
try:
    from google.colab import userdata
    secret_readers.append(userdata.get)
except ImportError:
    pass

required_key = "GOOGLE_API_KEY" if AI_PROVIDER == "gemini" else None
names = ["PEXELS_API_KEY", "HF_TOKEN"]
names.append("GOOGLE_API_KEY" if AI_PROVIDER == "gemini" else "VLLM_API_KEY")
for name in names:
    value = os.environ.get(name, "").strip()
    for read_secret in secret_readers:
        if value:
            break
        try:
            value = (read_secret(name) or "").strip()
        except Exception:
            pass  # Missing optional secrets must not block another provider.
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
del value

if required_key and not os.environ.get(required_key, "").strip():
    raise RuntimeError("Add GOOGLE_API_KEY to Colab/Kaggle Secrets or the environment for Gemini.")

if AI_PROVIDER == "vllm":
    from clipping.providers.config import vllm_config_values

    if START_LOCAL_VLLM:
        VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}"
        if not os.environ.get("VLLM_API_KEY", "").strip():
            from secrets import token_urlsafe
            os.environ["VLLM_API_KEY"] = token_urlsafe(32)
    settings = vllm_config_values({
        "vllm_base_url": VLLM_BASE_URL, "vllm_model": VLLM_MODEL,
        "vllm_context_limit": VLLM_CONTEXT_LIMIT,
        "local_max_tokens": VLLM_MAX_TOKENS, "vllm_timeout": VLLM_TIMEOUT,
    })
    if START_LOCAL_VLLM and settings["vllm_context_limit"] is None:
        raise ValueError("Set VLLM_CONTEXT_LIMIT before starting a local server.")
    if settings["vllm_context_limit"] and settings["local_max_tokens"] >= settings["vllm_context_limit"]:
        raise ValueError("VLLM_MAX_TOKENS must be smaller than VLLM_CONTEXT_LIMIT.")
    for name, field in {
        "VLLM_BASE_URL": "vllm_base_url", "VLLM_MODEL": "vllm_model",
        "VLLM_CONTEXT_LIMIT": "vllm_context_limit", "VLLM_MAX_TOKENS": "local_max_tokens",
        "VLLM_TIMEOUT": "vllm_timeout",
    }.items():
        os.environ[name] = "" if settings[field] is None else str(settings[field])
    print("vLLM configuration ready. GOOGLE_API_KEY is not required.")
else:
    # Preserve the existing Gemini .env workflow; vLLM authentication stays in memory.
    env_text = "# Auto-generated by the notebook\n"
    for name in ("GOOGLE_API_KEY", "PEXELS_API_KEY", "HF_TOKEN"):
        env_text += f"{name}={os.environ.get(name, '')}\n"
    Path(".env").write_text(env_text, encoding="utf-8")
    print("Gemini .env created. Secret values are not displayed.")


### Setup vLLM lokal (opsional)

Jika `START_LOCAL_VLLM = False`, sel berikut melewati instalasi dan startup. Jika `True`, jalankan Prepare sekali, lalu Start/Eval. Helper Kaggle memasang dependency aplikasi/server di luar kernel notebook. Jika setup/evaluasi gagal, perbaiki error sebelum clipping. Laporan: `outputs/vllm-eval-short.json`; log server: `logs/kaggle-vllm.log`.

Untuk video berikutnya dengan server yang sama, cukup ulangi sel cookie/pipeline. Gunakan Stop setelah selesai; hentikan server sebelum mengulangi Prepare.


In [ ]:
# Prepare isolated application and vLLM environments (opt-in)
LOCAL_VLLM_PREPARED = False
LOCAL_VLLM_READY = False
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    from scripts.kaggle_python import bootstrap_environment, ensure_python
    from scripts.kaggle_storage import check_storage, prepare_storage
    from scripts.kaggle_studio import runtime_environment

    # Reject unsupported GPUs before downloading Python or installing packages.
    subprocess.run(
        [sys.executable, "scripts/kaggle_vllm.py", "preflight", "--gpu-index", str(GPU_INDEX)],
        cwd=PROJECT_DIR, check=True,
    )
    workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else PROJECT_DIR
    storage = prepare_storage(PROJECT_DIR, workspace, mode="temporary")
    check_storage(storage, 1.0)
    cache_env = storage.cache_environment()
    BOOTSTRAP_PYTHON = ensure_python(storage.python_cache)
    setup_env = bootstrap_environment(storage.python_cache) | cache_env
    APP_ENV = storage.root / "kaggle-studio-cpu" / "venv"
    APP_PYTHON = str(APP_ENV / "bin/python")
    VLLM_ENV = storage.vllm_env
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_studio.py", "setup",
         "--venv", str(APP_ENV), "--torch-profile", "cpu"],
        cwd=PROJECT_DIR, env=setup_env, check=True,
    )
    check_storage(storage, 1.0)
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "setup", "--venv", str(VLLM_ENV)],
        cwd=PROJECT_DIR, env=setup_env, check=True,
    )
    check_storage(storage, 1.0)
    LOCAL_VLLM_PREPARED = True
    print("Application and vLLM environments ready; the notebook kernel is unchanged.")
else:
    print("Local vLLM installation skipped.")


In [ ]:
# Start local vLLM and evaluate one short transcript (opt-in)
LOCAL_VLLM_READY = False
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    if not LOCAL_VLLM_PREPARED:
        raise RuntimeError("Run the isolated vLLM preparation cell successfully first.")
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "start",
         "--venv", str(VLLM_ENV), "--gpu-index", str(GPU_INDEX),
         "--model", os.environ["VLLM_MODEL"], "--port", str(VLLM_PORT),
         "--context", os.environ["VLLM_CONTEXT_LIMIT"], "--gpu-memory-utilization", "0.85"],
        cwd=PROJECT_DIR, env=bootstrap_environment(storage.python_cache) | cache_env, check=True,
    )
    EVAL_OUTPUT = PROJECT_DIR / "outputs" / "vllm-eval-short.json"
    subprocess.run(
        [APP_PYTHON, "scripts/eval_local_llm.py",
         "--transcript", "tests/fixtures/eval_transcript_short.txt",
         "--clips", "1", "--output", str(EVAL_OUTPUT)],
        cwd=PROJECT_DIR, env=runtime_environment(APP_ENV) | cache_env, check=True,
    )
    LOCAL_VLLM_READY = True
    print(f"vLLM evaluation passed. Report: {EVAL_OUTPUT}")
else:
    print("Local start skipped. For vLLM, the configured external server must already be running.")


## YouTube Cookies (Opsional)

Jika YouTube meminta login atau menolak unduhan tanpa autentikasi, gunakan ekspor **cookies.txt berformat Netscape** dari browser. Jalankan setup proyek terlebih dahulu, lalu unggah file dan isi `YOUTUBE_COOKIES_FILE` pada sel berikut:

- **Colab:** unggah melalui sidebar Files, lalu gunakan `/content/cookies.txt`.
- **Kaggle:** lampirkan Input privat, lalu salin path filenya (contoh `/kaggle/input/my-private-cookies/cookies.txt`).
- **Lokasi lain:** gunakan path absolut atau relatif terhadap folder kerja notebook. Path dengan spasi didukung.

Nilai awal dibaca dari environment variable `OSC_YOUTUBE_COOKIES_FILE`. Ganti dengan string path untuk memilih file secara manual. Jika kosong, deteksi otomatis `youtube_cookies.txt` di lokasi lama tetap berjalan; tanpa file yang terdeteksi, unduhan dicoba tanpa cookies. Path yang diisi tetapi tidak tersedia atau menunjuk ke folder akan menghasilkan error.

Jalankan ulang sel cookie setelah mengganti path. Simpan file cookie secara privat dan jangan sertakan dalam output notebook yang dibagikan. Cookie dapat membantu autentikasi, tetapi tidak menjamin unduhan berhasil.


In [ ]:
# YouTube Cookies Setup (Optional)
import os
import shlex
import shutil

# Replace the value with your uploaded file path, or leave empty for auto-detection.
# Example: YOUTUBE_COOKIES_FILE = "/content/cookies.txt"
YOUTUBE_COOKIES_FILE = os.environ.get("OSC_YOUTUBE_COOKIES_FILE", "")

YT_COOKIES_ARG = ""
COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

if YOUTUBE_COOKIES_FILE.strip():
    source_cookie = os.path.abspath(os.path.expanduser(YOUTUBE_COOKIES_FILE.strip()))
    if not os.path.isfile(source_cookie):
        raise FileNotFoundError(
            "YOUTUBE_COOKIES_FILE must point to an existing cookies.txt file. "
            "Upload it to Colab or attach a private Kaggle Input, then copy its file path."
        )
else:
    source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        # Copy contents only: Kaggle Input files may be read-only.
        shutil.copyfile(source_cookie, COOKIES_PATH)
    print(f"YouTube cookies active: {COOKIES_PATH}")
    YT_COOKIES_ARG = f"--yt-cookies {shlex.quote(COOKIES_PATH)}"
else:
    print("youtube_cookies.txt not found. The download will run without cookies.")


In [ ]:
# Execute Test
# Define your parameters here as global variables
URL_YOUTUBE = "https://www.youtube.com/watch?v=Ip6pCjWp4lk"
JUMLAH_CLIP = 1
RASIO = "9:16"
FONT_STYLE = "DEFAULT"
WHISPER_MODEL = "large-v3"
WHISPER_DEVICE = "cuda"
WHISPER_COMPUTE_TYPE = "float16"

# Run the clipping pipeline with the selected provider
import shlex

if AI_PROVIDER == "vllm" and START_LOCAL_VLLM and not LOCAL_VLLM_READY:
    raise RuntimeError("Run the local vLLM Start/Eval cell successfully before clipping.")

run_whisper_device = WHISPER_DEVICE
run_whisper_compute = WHISPER_COMPUTE_TYPE
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    run_whisper_device, run_whisper_compute = "cpu", "int8"
    print("Local vLLM uses the GPU; Whisper will use CPU/int8.")

command = [
    APP_PYTHON, "main.py",
    "--url", URL_YOUTUBE, "--clips", str(JUMLAH_CLIP), "--ratio", RASIO,
    "--font-style", FONT_STYLE, "--hook-duration", str(3),
    "--words-per-sub", str(5), "--ai-provider", AI_PROVIDER,
    "--whisper-model", WHISPER_MODEL, "--whisper-device", run_whisper_device,
    "--whisper-compute-type", run_whisper_compute,
]
if AI_PROVIDER == "vllm":
    # Process long transcripts in windows instead of requiring one giant prompt.
    command.append("--vllm-chunk-transcript")
command.extend(shlex.split(YT_COOKIES_ARG))
pipeline_env = (runtime_environment(APP_ENV) | cache_env
                if APP_ENV is not None else os.environ.copy())
subprocess.run(command, cwd=PROJECT_DIR, env=pipeline_env, check=True)


### Hentikan vLLM lokal setelah selesai

Biarkan `STOP_LOCAL_VLLM = False` selama memproses video. Ubah menjadi `True` lalu jalankan setelah selesai; server eksternal tidak dihentikan.


In [ ]:
# Stop the managed local server only when you are finished
STOP_LOCAL_VLLM = False  # Set True and run this cell after all clipping jobs are complete.
if STOP_LOCAL_VLLM and START_LOCAL_VLLM:
    subprocess.run(
        [sys.executable, "scripts/kaggle_vllm.py", "stop"], cwd=PROJECT_DIR, check=True,
    )
    LOCAL_VLLM_READY = False
    print("Managed local vLLM stopped. Re-run Start/Eval before another vLLM job.")


# 2. Update to Youtube

In [ ]:
# do your magic here